In [ ]:
import os
import glob
import pandas as pd
# Obtener todos los archivos CSV de la carpeta

df = pd.read_csv('gexV9/experimento-batch/integration_metrics.csv')

# Print header with column names
header = ' & '.join(df.columns)
print(header + ' \\\\')
print('\\hline')

# Print each row
for _, row_data in df.iterrows():
    formatted_values = []
    for col, val in zip(df.columns, row_data):
        if col == 'logreg_covgap':
            # Format logreg_covgap with 3 decimals
            formatted_values.append(f"{val:.3f}" if isinstance(val, (int, float)) and pd.notna(val) else str(val))
        elif isinstance(val, (int, float)) and pd.notna(val):
            # Format other numeric values with 3 decimals
            formatted_values.append(f"{val:.2f}")
        else:
            # Keep non-numeric values as strings
            formatted_values.append(str(val))
    
    values = ' & '.join(formatted_values)
    print(values + ' \\\\')

In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt

import numpy as np
import scanpy as sc
import pandas as pd
import os


path = "../../scripts/experiments/diabetic-kidney/experimento-batch-2"

adata_post = sc.read_h5ad(os.path.join(path, "adata_processed_batch_embeddings.h5ad"))
#adata_post.obsm["X_ConfTr"] = adata_1.obsm["X_ConfTr"]

# ---------- CONFIG (edit these) ----------
methods = ["scanvi", "combat", "scanorama", "scvi", "harmony", "conftr"]
variables = ["tissue", "assay", "cell_type", "development_stage", "disease", "batch"]

# Map methods -> AnnData .obsm key that holds the integrated representation used for neighbors
# Adjust to your dataset if needed:
rep_key_map = {
    "scanvi":    "X_scanvi",
    "combat":    "X_pca_combat",
    "scanorama": "X_scanorama",
    "scvi":      "X_scvi",       # often X_scVI or X_latent depending on scvi-tools version
    "harmony":   "X_harmony",    # sometimes X_pca_harmony
    "Conftr":    "X_conftr"
}

# UMAP / plotting params
N_NEIGHBORS   = 15
MIN_DIST      = 0.5
RANDOM_STATE  = 0
POINT_SIZE    = 2.0            # tune for your n_cells (smaller for very large datasets)
NA_COLOR      = "lightgray"    # how to show missing categories
FIG_DPI       = 300            # hi-res for print
COL_TITLE_FZ  = 11
ROW_LABEL_FZ  = 11
SUPTITLE_FZ   = 13

# ---------- HELPERS ----------
def _find_rep_key(adata, method, rep_key_map):
    """Find a representation key in .obsm to use for neighbors for a given method."""
    # 1) explicit map if provided
    if method in rep_key_map and rep_key_map[method] in adata.obsm:
        return rep_key_map[method]
    # 2) try common variants + fuzzy fallback
    candidates_by_method = {
        "scanvi":    ["X_scanvi", "X_scANVI", "X_scanvi_latent"],
        "scvi":      ["X_scvi", "X_scVI", "X_latent", "X_scvi_latent"],
        "harmony":   ["X_harmony", "X_pca_harmony"],
        "combat":    ["X_combat", "X_pca_combat"],
        "scanorama": ["X_scanorama", "X_pca_scanorama"],
    }
    for key in candidates_by_method.get(method, []):
        if key in adata.obsm:
            return key
    # 3) fuzzy: any obsm key that contains the method name
    lower_method = method.lower()
    for k in adata.obsm_keys():
        if lower_method in k.lower():
            return k
    # 4) final fallback
    if "X_pca" in adata.obsm:
        return "X_pca"
    raise KeyError(f"Could not find a representation in .obsm for method '{method}'. "
                   f"Please update rep_key_map.")

def _ensure_umap_for_method(adata, method, rep_key, n_neighbors, min_dist, random_state):
    """Compute UMAP coords for this method once and store at .obsm[f'X_umap_{method}']."""
    umap_key = f"X_umap_{method}"
    if umap_key in adata.obsm:
        return umap_key  # already computed
    neigh_key = f"neighbors_{method}"
    sc.pp.neighbors(
        adata,
        n_neighbors=n_neighbors,
        use_rep=rep_key,
        key_added=neigh_key
    )
    sc.tl.umap(
        adata,
        neighbors_key=neigh_key,
        min_dist=min_dist,
        random_state=random_state
    )
    # Move the coords to a method-specific slot, then clean up the temporary default key
    adata.obsm[umap_key] = adata.obsm["X_umap"].copy()
    del adata.obsm["X_umap"]
    return umap_key

def plot_integration_panel(
    adata,
    methods,
    variables,
    rep_key_map,
    n_neighbors=15,
    min_dist=0.5,
    random_state=0,
    point_size=2.0,
    na_color="lightgray",
    fig_dpi=600,
    suptitle="Integrated UMAPs across methods",
    draw_col_separators=True,
    sep_color="0.85",
    sep_lw=0.6,
    sep_alpha=1.0
):
    nrows, ncols = len(variables), len(methods)
    # Proportional sizing: tweak base cell size for your density
    cell_w, cell_h = 4.2, 3.8
    fig_w, fig_h = cell_w * ncols, cell_h * nrows

    fig, axes = plt.subplots(nrows, ncols, figsize=(fig_w, fig_h), dpi=fig_dpi, constrained_layout=True)
    if nrows == 1 and ncols == 1:
        axes = np.array([[axes]])
    elif nrows == 1:
        axes = axes.reshape(1, -1)
    elif ncols == 1:
        axes = axes.reshape(-1, 1)

    # Compute (or reuse) UMAP per method
    umap_basis_for = {}
    for method in methods:
        rep_key = _find_rep_key(adata, method, rep_key_map)
        umap_key = _ensure_umap_for_method(
            adata, method, rep_key, n_neighbors, min_dist, random_state
        )
        # Scanpy convention: basis is the suffix after 'X_'
        basis = umap_key.replace("X_", "")
        umap_basis_for[method] = basis

    # Plot grid
    for j, method in enumerate(methods):
        basis = umap_basis_for[method]
        for i, var in enumerate(variables):
            ax = axes[i, j]
            show_legend = (j == ncols - 1)  # only on the last column for each row
            # Plot using scanpy's embedding plotter so categorical palettes are respected
            sc.pl.embedding(
                adata,
                basis=basis,
                color=var,
                ax=ax,
                show=False,
                s=point_size,
                frameon=False,
                na_color=na_color,
                legend_loc="right margin" if show_legend else None
            )
            # Column titles on the top row
            if i == 0:
                ax.set_title(method, fontsize=COL_TITLE_FZ, fontweight="bold", pad=6)
            # Row labels on the leftmost column
            if j == 0:
                ax.set_ylabel(var, fontsize=ROW_LABEL_FZ)
            else:
                ax.set_ylabel("")

    fig.suptitle(suptitle, fontsize=SUPTITLE_FZ, y=1.02)

    # === LÍNEAS VERTICALES ENTRE COLUMNAS ===
    if draw_col_separators and ncols > 1:
        fig.canvas.draw()

        # Extremos verticales de la parrilla (en coords de figura)
        y_top = max(axes[i, 0].get_position(fig).y1 for i in range(nrows))
        y_bot = min(axes[i, 0].get_position(fig).y0 for i in range(nrows))

        # Para cada frontera entre columnas j | j+1
        for j in range(ncols - 1):
            # Derecho de la col j e izquierdo de la col j+1, promediados sobre filas
            x_right_j = max(axes[i, j].get_position(fig).x1 for i in range(nrows))
            x_left_next = min(axes[i, j+1].get_position(fig).x0 for i in range(nrows))
            x_mid = 0.5 * (x_right_j + x_left_next)

            line = plt.Line2D(
                [x_mid, x_mid], [y_bot, y_top],
                transform=fig.transFigure,
                color=sep_color, linewidth=sep_lw, alpha=sep_alpha,
                solid_capstyle="butt", zorder=3, clip_on=False
            )
            fig.add_artist(line)



    return fig

# ---------- RUN IT ----------
# Use your integrated AnnData (e.g., adata_post)
fig = plot_integration_panel(
    adata=adata_post,
    methods=methods,
    variables=variables,
    rep_key_map=rep_key_map,
    n_neighbors=N_NEIGHBORS,
    min_dist=MIN_DIST,
    random_state=RANDOM_STATE,
    point_size=POINT_SIZE,
    na_color=NA_COLOR,
    fig_dpi=FIG_DPI,
    suptitle="Integrated UMAPs across methods",
    draw_col_separators=True,  # activa las líneas
    sep_color="0.85",          # gris clarito (también puedes usar "#D0D0D0")
    sep_lw=0.6,                # grosor fino
    sep_alpha=1.0
)

# Save both PDF (vector) and PNG (raster)

fig.savefig(os.path.join(path, "umap_integration_panel.pdf"), bbox_inches="tight")
fig.savefig(os.path.join(path, "umap_integration_panel.png"), bbox_inches="tight")
plt.show()
